In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

/Users/lgoncalves/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
import os
from getpass import getpass

api_key = getpass("Cole sua OPENAI_API_KEY e pressione Enter: ").strip()
if not api_key:
    raise ValueError("Adicione sua OPENAI_API_KEY para continuar.")

os.environ["OPENAI_API_KEY"] = api_key

In [3]:
# Carregar modelos openai - embeddings e chat
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(
    model="gpt-6-luna",
    max_completion_tokens=2000,
    reasoning_effort="low",
)


In [4]:
#load do arquivo
pdf_link = "os-sertoes.pdf"

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

In [5]:
# Separar em chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True,
)
chunks = text_splitter.split_documents(pages)


In [6]:
# Salvar os chunks no vectorDB
vectordb = Chroma(embedding_function=embeddings_model, persist_directory="naiveDB")

/var/folders/vn/hc513xd51dx9541yf23mg48r0000gp/T/ipykernel_92347/2868498988.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(embedding_function=embeddings_model, persist_directory="naiveDB")


In [7]:
# Carregar o DB
naive_retriever = vectordb.as_retriever(search_kwargs={"k": 10})

In [8]:
from getpass import getpass

cohere_api_key = getpass("Cole sua COHERE_API_KEY e pressione Enter: ").strip()
if not cohere_api_key:
    raise ValueError("Adicione sua COHERE_API_KEY para continuar.")

os.environ["COHERE_API_KEY"] = cohere_api_key

In [9]:
rerank = CohereRerank(model="rerank-v3.5", top_n=3)

# Criar o retriever contextual
compression_retriever = ContextualCompressionRetriever(
    base_retriever=naive_retriever,
    base_compressor=rerank,
)

In [10]:
TEMPLATE = """
Você é um especialista em literatura brasileira. Responda a pergunta abaixo utilizando o contexto informado.
Query: {question}

Contexto:
{context}

Resposta:
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [11]:
setup_retrival = RunnableParallel({
    "question": RunnablePassthrough(),
    "context": compression_retriever
})

output_parser = StrOutputParser()

compressor_retrival_chain = setup_retrival | rag_prompt | llm | output_parser

In [13]:
user_question = input("Digite sua pergunta: ")
compressor_retrival_chain.invoke(user_question)

'Em **_Os Sertões_** (1902), Euclides da Cunha transforma a Guerra de Canudos em uma crítica ampla ao Brasil republicano. A obra é organizada em três partes — **“A Terra”, “O Homem” e “A Luta”** — e mostra como o conflito não pode ser entendido apenas como uma campanha militar contra um movimento religioso.\n\n- **Abandono do sertão:** Euclides descreve a pobreza, a seca e o isolamento da população sertaneja. O contraste entre o litoral urbanizado e o interior esquecido revela um país profundamente desigual, cujas elites conheciam pouco a realidade de grande parte da população.\n- **Violência do Estado e da República:** A destruição de Canudos expõe a brutalidade da repressão e a incapacidade do governo de compreender o movimento. A República, que se apresentava como moderna e civilizadora, trata os sertanejos como inimigos a eliminar.\n- **Crítica à incompreensão entre Brasil urbano e Brasil sertanejo:** A campanha é alimentada por preconceitos e pela ideia de que Canudos ameaçava a o